# Figure 2: genetic diversity of *An. dirus* s.l.

(a) Runs of homozygosity (ROH) per sample and (b) nucleotide diversity (π) and Tajima's D per cohort, with comparator populations of *An. minimus*, *An. stephensi*, *An. funestus* and *An. gambiae*. *An. dirus*, *An. minimus* and *An. stephensi* statistics are computed with the Adir1.0, Amin1.0 and As1.0 data via the `malariagen_data` API.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import malariagen_data
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm

## Setup

In [ ]:
# Results cache shared by all notebooks, so expensive analyses (PCA, ROH, GWSS)
# are only computed once
RESULTS_CACHE = Path("~/sea_vector_cache").expanduser()
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)


# In malariagen_data 15.10.0, Adir1 `variant_allele` arrays are object dtype, which
# zarr cannot write to the results cache (ValueError: missing object_codec for
# object array). Cast them to bytes before caching.
class Adir1(malariagen_data.Adir1):
    def results_cache_set(self, *, name, params, results):
        results = {k: v.astype("S") if v.dtype == object else v for k, v in results.items()}
        super().results_cache_set(name=name, params=params, results=results)

In [ ]:
adir1 = Adir1(results_cache=str(RESULTS_CACHE), show_progress=False)
df_samples = adir1.sample_metadata()

# An. minimus cohorts (PV/RK1/RK2/TD) are the population assignments of
# St Laurent et al. 2022, attached as the `cam_population` metadata column
amin1 = malariagen_data.Amin1(results_cache=str(RESULTS_CACHE), show_progress=False)
amin1.add_extra_metadata(pd.read_csv("../supplementary_data/amin1_cohorts.csv"))

AMIN1_COHORTS = {pop: f"cam_population == '{pop}'" for pop in ["TD", "PV", "RK1", "RK2"]}

as1 = malariagen_data.As1(results_cache=str(RESULTS_CACHE), show_progress=False)

In [ ]:
# An. dirus s.l. cohorts, defined from the PCA in Figure 1B. Four samples from
# Cambodian site RS23 cluster with An. baimaii.
RS23_BAIMAII = ["S1214", "S1312", "S1047", "S1371"]

ADIR1_COHORTS = {
    "pca_groupa": f"country == 'Thailand' or partner_sample_id in {RS23_BAIMAII}",
    "pca_groupb": f"country == 'Cambodia' and partner_sample_id not in {RS23_BAIMAII}",
    "pca_groupc": "country == 'Bangladesh'",
}

ADIR1_COHORT_LABELS = {
    "pca_groupa": "baimaii-Cambodia/Thailand",
    "pca_groupb": "dirus_ss-Cambodia",
    "pca_groupc": "baimaii-Bangladesh",
}

# An. stephensi comparator cohorts (Dennis et al. 2025)
AS1_COHORTS = {
    "stephensi-Afghanistan/Pakistan": "country in ['Afghanistan', 'Pakistan']",
    "stephensi-Sudan": "country == 'Sudan' and location != 'PortSudan2018'",
}

COUNTRY_COLOURS = {
    "Bangladesh": "#e41a1c",
    "Cambodia": "#377eb8",
    "Thailand": "#4daf4a",
}

## Runs of homozygosity

ROH are inferred per sample and per contig with `roh_hmm`, and runs shorter than 100 kb are discarded. fROH is the total length of ROH divided by the total length of the 79 contigs.

This is 540 samples × 79 contigs of HMM fits. Results are cached in `RESULTS_CACHE`, so the first run is slow (hours) and later runs are fast.

In [ ]:
ROH_MIN_LENGTH = 100_000
genome_length = sum(len(adir1.genome_sequence(c)) for c in adir1.contigs)


def sample_roh(sample_id):
    return pd.concat([adir1.roh_hmm(sample=sample_id, region=c) for c in adir1.contigs])


with ThreadPoolExecutor(max_workers=8) as pool:
    df_roh = pd.concat(
        tqdm(pool.map(sample_roh, df_samples["sample_id"]), total=len(df_samples))
    )

In [ ]:
df_roh_long = df_roh.query("roh_length >= @ROH_MIN_LENGTH")

df_roh_summary = (
    df_roh_long.groupby("sample_id")["roh_length"]
    .agg(nROH="size", lROH="sum")
    .reindex(df_samples["sample_id"], fill_value=0)
    .reset_index()
    .merge(df_samples, on="sample_id")
)
df_roh_summary["fROH"] = df_roh_summary["lROH"] / genome_length
df_roh_summary.to_csv(FIGURES_DIR / "fig2a_roh.csv", index=False)
df_roh_summary.groupby("country")[["nROH", "fROH"]].describe().T

## π and Tajima's D

### *An. dirus* s.l., *An. minimus* and *An. stephensi*

All are computed with `cohort_diversity_stats` on the longest contig of each assembly (KB672490 for Adir1, KB663610 for Amin1, 2RL for As1). Cohort sizes are 30, except *An. minimus* which uses 28 (the size of the smallest population, RK2). 95% confidence intervals are from 200 block-jackknife replicates (`n_jack=200`, the API default).

In [ ]:
STATS_COLS = ["cohort", "theta_pi", "theta_pi_ci_low", "theta_pi_ci_upp", "tajima_d", "tajima_d_ci_low", "tajima_d_ci_upp"]

df_dirus_thetas = pd.DataFrame([
    adir1.cohort_diversity_stats(
        cohort=(ADIR1_COHORT_LABELS[name], query), cohort_size=30, region=adir1.contigs[0]
    )
    for name, query in ADIR1_COHORTS.items()
])

df_minimus_thetas = pd.DataFrame([
    amin1.cohort_diversity_stats(
        cohort=(f"minimus-{name}", query), cohort_size=28, region=amin1.contigs[0]
    )
    for name, query in AMIN1_COHORTS.items()
])

df_stephensi_thetas = pd.DataFrame([
    as1.cohort_diversity_stats(cohort=(name, query), cohort_size=30, region="2RL")
    for name, query in AS1_COHORTS.items()
])

### *An. gambiae* and *An. funestus*

Tanzanian comparator populations, computed with the Ag3 and Af1 APIs.

In [ ]:
ag3 = malariagen_data.Ag3(results_cache=str(RESULTS_CACHE), show_progress=False)
df_gambiae = ag3.diversity_stats(
    sample_query="release == '3.0' and country == 'Tanzania'",
    cohorts="admin1_year",
    cohort_size=10,
    region="3L:15,000,000-41,000,000",
    site_mask="gamb_colu_arab",
    site_class="CDS_DEG_4",
).query("cohort == 'TZ-05_gamb_2015'").assign(cohort="gambiae-Tanzania")

af1 = malariagen_data.Af1(results_cache=str(RESULTS_CACHE), show_progress=False)
df_funestus = af1.diversity_stats(
    sample_query="country == 'Tanzania' and admin1_name == 'Kagera'",
    sample_sets=[
        "1236-VO-TZ-OKUMU-OKFR-TZ-2008",
        "1236-VO-TZ-OKUMU-VMF00248",
        "1236-VO-TZ-OKUMU-VMF00252",
        "1236-VO-TZ-OKUMU-VMF00261",
        "1236-VO-TZ-OKUMU-VMF00090",
    ],
    cohorts="admin1_year",
    cohort_size=10,
    region="2RL:15,000,000-41,000,000",
    site_class="CDS_DEG_4",
).assign(cohort="funestus-Tanzania")

In [ ]:
df_thetas = pd.concat(
    [d[STATS_COLS] for d in [df_dirus_thetas, df_minimus_thetas, df_stephensi_thetas, df_funestus, df_gambiae]],
    ignore_index=True,
)
df_thetas[STATS_COLS[1:]] = df_thetas[STATS_COLS[1:]].astype(float)
df_thetas.to_csv(FIGURES_DIR / "fig2b_diversity_stats.csv", index=False)
df_thetas

## Figure 2

(A) ROH per sample; (B) π (top) and Tajima's D (bottom) per cohort.

In [ ]:
COHORT_PLOT_ORDER = [
    "baimaii-Bangladesh",
    "dirus_ss-Cambodia",
    "baimaii-Cambodia/Thailand",
    "minimus-TD",
    "minimus-PV",
    "minimus-RK1",
    "minimus-RK2",
    "stephensi-Afghanistan/Pakistan",
    "stephensi-Sudan",
    "funestus-Tanzania",
    "gambiae-Tanzania",
]

COHORT_COLOURS = dict(zip(COHORT_PLOT_ORDER, [
    COUNTRY_COLOURS["Bangladesh"],
    COUNTRY_COLOURS["Cambodia"],
    COUNTRY_COLOURS["Thailand"],
    *["#fdbf6f"] * 4,
    *["#fb9a99"] * 2,
    "#cab2d6",
    "#999999",
]))


def plot_roh(ax, df):
    sns.scatterplot(
        data=df, x="fROH", y="nROH", hue="country",
        palette=COUNTRY_COLOURS, hue_order=list(COUNTRY_COLOURS),
        s=80, alpha=0.6, edgecolor="white", linewidth=0.5, ax=ax,
    )
    sns.despine(ax=ax, offset=10)
    ax.set_xlabel("fROH", fontsize=14)
    ax.set_ylabel("nROH", fontsize=14)
    ax.tick_params(labelsize=12, width=0.8, length=5)
    ax.legend(title="Country", fontsize=12, title_fontsize=13, loc="lower right", frameon=False)


def plot_thetas(ax, df, stat, symbol, show_xticks=True):
    df_plot = df.set_index("cohort").loc[COHORT_PLOT_ORDER]
    for i, (cohort, row) in enumerate(df_plot.iterrows()):
        ax.errorbar(
            i, row[stat],
            yerr=[[row[stat] - row[f"{stat}_ci_low"]], [row[f"{stat}_ci_upp"] - row[stat]]],
            fmt="o", color=COHORT_COLOURS[cohort], markersize=7, capsize=5, linewidth=2,
        )
    sns.despine(ax=ax)
    ax.spines["left"].set_position(("outward", 10))
    ax.set_xticks(range(len(COHORT_PLOT_ORDER)))
    ax.set_xticklabels(COHORT_PLOT_ORDER if show_xticks else [], rotation=45, ha="right", fontsize=11)
    ax.yaxis.set_major_locator(matplotlib.ticker.MaxNLocator(nbins=4))
    ax.tick_params(axis="y", labelsize=12)
    ax.set_ylabel(symbol, fontsize=14)

In [ ]:
sns.set_style("ticks")
fig = plt.figure(figsize=(13, 3.9))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.15], wspace=0.3, hspace=0.12)
ax_roh = fig.add_subplot(gs[:, 0])
ax_pi = fig.add_subplot(gs[0, 1])
ax_td = fig.add_subplot(gs[1, 1], sharex=ax_pi)

plot_roh(ax_roh, df_roh_summary)
plot_thetas(ax_pi, df_thetas, "theta_pi", "π", show_xticks=False)
plot_thetas(ax_td, df_thetas, "tajima_d", "Tajima's D")
plt.setp(ax_pi.get_xticklabels(), visible=False)

for ax, label, x in [(ax_roh, "A", -0.12), (ax_pi, "B", -0.13)]:
    ax.text(x, 1.02, label, transform=ax.transAxes, fontsize=18, fontweight="bold", va="bottom")

fig.savefig(FIGURES_DIR / "fig2.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "fig2.png", bbox_inches="tight", dpi=200)